# Agentic RAG with LangGraph

This notebook demonstrates building an Agentic RAG system using LangGraph with autonomous agents.

## What You'll Learn

1. **Agent Architecture** - How agents make decisions
2. **LangGraph Basics** - Building graphs with nodes and edges
3. **Tool Use** - Equipping agents with capabilities
4. **Conditional Logic** - Dynamic decision paths
5. **Self-Correction** - Agent evaluates and retries

## What Makes RAG "Agentic"?

Classic RAG is a **fixed pipeline**: Query → Retrieve → Generate → Done

Agentic RAG is **dynamic**: The agent can:

- Plan its approach before acting
- Evaluate if retrieved info is sufficient
- Decide to retry with different queries
- Use multiple tools beyond just retrieval
- Iterate until satisfied with the answer

## Classic RAG vs Agentic RAG

| Aspect | Classic RAG | Agentic RAG |
|--------|-------------|-------------|
| Flow | Fixed pipeline | Dynamic, conditional |
| Self-correction | No | Yes (evaluate & retry) |
| Tools | Just retrieval | Multiple tools |
| Planning | None | Creates execution plan |
| Complexity | Lower | Higher |
| Latency | Lower | Higher (multiple steps) |

## Architecture: Agent Loop

```
    ┌─────────────────────────────────────────────┐
    │                                             │
    ▼                                             │
┌────────┐    ┌─────────┐    ┌──────────┐    ┌─────┴────┐
│Planner │───▶│Retrieve │───▶│ Evaluate │───▶│ Sufficient│
└────────┘    └─────────┘    └──────────┘    └─────┬────┘
      ▲                                            │
      │              ┌──────────┐                  │
      └──────────────│ Generate │◀─────────────────┘
                    └──────────┘
                         │
                         ▼
                    ┌─────────┐
                    │  Answer │
                    └─────────┘
```

## Setup

In [ ]:
!pip install langgraph langchain langchain-openai langchain-community
!pip install grandalf

## Import Libraries

In [1]:
import os
from langgraph.graph import StateGraph, END
from langchain_community.document_loaders import TextLoader
from langchain_community.tools import Tool
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from pydantic import BaseModel
from typing import List, TypedDict, Optional
import json


## Define Agent State

**What:** Defining the state that flows through the agent graph.

**Why:** The agent needs to track information across multiple steps. LangGraph uses TypedDict for type-safe state management.

## State Fields Explained

| Field | Type | Purpose |
|-------|------|---------|
| `question` | str | The user's query |
| `plan` | List[str] | Steps to answer the question |
| `retrieved_docs` | List | Documents from retrieval |
| `evaluation` | str | Is retrieved info sufficient? |
| `final_answer` | str | The generated answer |
| `iterations` | int | How many times we've retried |

> Think of state as "working memory" that each node can read from and write to.

In [2]:
class AgentState(TypedDict):
    """State for the Agentic RAG agent."""
    question: str
    plan: List[str]
    retrieved_docs: List
    evaluation: str
    final_answer: Optional[str]
    iterations: int

## Load Document

In [3]:
# Create sample document
sample_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that enhances 
Large Language Models by retrieving relevant information from external 
knowledge bases. RAG addresses key limitations of LLMs including 
hallucinations, knowledge cutoff, and lack of source attribution.

The basic RAG pipeline consists of three components:
1. Retrieval: Finding relevant documents
2. Augmentation: Adding context to the prompt
3. Generation: Producing the final response

RAG is particularly useful for:
- Question answering systems
- Knowledge base chatbots
- Enterprise search
- Customer support automation
"""

# Save sample document
with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

# Load document
loader = TextLoader("sample_doc.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


## Create Tools

**What:** Defining the capabilities the agent can use.

**Why:** Tools extend the agent's abilities beyond just retrieval. Each tool is a function the agent can call.

## Tools in This Agent

| Tool | Function | When Used |
|------|----------|------------|
| `vector_search` | Search knowledge base | To find relevant documents |
| `analyze_context` | Evaluate if context answers question | After retrieval |

## Tool Design Pattern

```python
def my_tool(query: str) -> str:
    """Description (important for agent!)"""
    # Tool logic here
    return result

Tool(
    name="tool_name",
    func=my_tool,
    description="What the tool does (used by LLM to decide)"
)
```

> **Key:** The description is critical! The LLM uses it to decide WHEN to use each tool.

In [4]:
# Initialize components
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Create vector store (using existing documents)
vectorstore = Chroma.from_documents(
    documents=documents,  # From previous notebook
    embedding=embeddings
)

def vector_search(query: str) -> str:
    """Search the knowledge base."""
    docs = vectorstore.similarity_search(query, k=4)
    return "\n\n".join([f"Doc {i+1}: {doc.page_content[:200]}" for i, doc in enumerate(docs)])

def analyze_context(query: str, context: str) -> str:
    """Analyze if context answers the question."""
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nDoes this answer the question?"
    return llm.invoke(prompt)

tools = [
    Tool(
        name="vector_search",
        func=vector_search,
        description="Search internal knowledge base"
    ),
    Tool(
        name="analyze_context",
        func=analyze_context,
        description="Analyze if retrieved context answers the question"
    )
]

## Define Agent Nodes

**What:** Creating the functions that perform each step of the agent's work.

**Why:** Each node in the LangGraph performs a specific task. Nodes are just Python functions that take state and return state updates.

## Node Functions Explained

| Node | Function | What It Does |
|-----|----------|---------------|
| `planner_node` | Creates execution plan | Decides HOW to answer the question |
| `retriever_node` | Searches documents | Finds relevant information |
| `evaluator_node` | Judges quality | Decides if retrieved info is sufficient |
| `generator_node` | Creates answer | Produces final response |

## How Nodes Work

```python
def node_name(state: AgentState) -> AgentState:
    """Node documentation"""
    # Read from state
    question = state['question']
    
    # Do work...
    result = do_something(question)
    
    # Return updates (only what changed!)
    return {"output_field": result}
```

> **Important:** Nodes return state updates (dict), not the entire state. Only include fields that changed.

In [5]:
def planner_node(state: AgentState) -> AgentState:
    """Plan retrieval strategy."""
    prompt = f"""Given this question: {state['question']}\n\nCreate a plan to answer it.\nRespond as a JSON list of steps."""
    response = llm.invoke(prompt)
    try:
        plan = json.loads(response)
    except:
        plan = ["Search knowledge base", "Analyze results", "Generate answer"]
    return {"plan": plan, "iterations": state.get("iterations", 0)}

def retriever_node(state: AgentState) -> AgentState:
    """Execute retrieval."""
    docs = vectorstore.similarity_search(state['question'], k=4)
    return {"retrieved_docs": docs}

def evaluator_node(state: AgentState) -> AgentState:
    """Evaluate retrieval quality."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Question: {state['question']}\n\nContext:\n{context}\n\nIs this sufficient?\n\nRespond: 'sufficient', 'need_more', or 'refine'"""
    evaluation = llm.invoke(prompt).content.strip().lower()
    return {"evaluation": evaluation}

def generator_node(state: AgentState) -> AgentState:
    """Generate final answer."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Based on the context, answer the question.\n\nContext:\n{context}\n\nQuestion: {state['question']}\n\nAnswer:"""
    answer = llm.invoke(prompt)
    return {"final_answer": answer}

## Build the Graph

**What:** Connecting nodes with edges to create the agent workflow.

**Why:** The graph defines HOW the agent flows from one step to the next.

## Graph Components

| Component | Description |
|-----------|-------------|
| `add_node()` | Adds a processing step |
| `add_edge()` | Creates a direct connection |
| `add_conditional_edges()` | Dynamic routing based on logic |
| `set_entry_point()` | Where the graph starts |

## The should_continue Function

This function determines the flow after evaluation:

```python
def should_continue(state: AgentState) -> str:
    if state.get("evaluation") == "need_more":
        if state.get("iterations", 0) < 2:  # Max 2 retries
            return "retry"  # Go back to planner
    return "generate"  # Good enough, generate answer
```

## Why Conditional Edges?

This enables **self-correction** - the agent can:
- Decide retrieval wasn't enough → retry with new plan
- Decide info is sufficient → proceed to generate
- Limit retries to prevent infinite loops (max 2 here)

> This is what makes it "agentic" - the agent evaluates and decides!

In [6]:
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("retrieve", retriever_node)
workflow.add_node("evaluate", evaluator_node)
workflow.add_node("generate", generator_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "retrieve")
workflow.add_edge("retrieve", "evaluate")

def should_continue(state: AgentState) -> str:
    """Decide whether to continue or finish."""
    if state.get("evaluation", "") == "need_more":
        if state.get("iterations", 0) < 2:
            return "retry"
    return "generate"

workflow.add_conditional_edges(
    "evaluate",
    should_continue,
    {"retry": "planner", "generate": "generate"}
)

workflow.add_edge("generate", END)

app = workflow.compile()

## Run Agentic RAG

**What:** Executing a query through the agent graph.

## What Happens During Execution

1. **Start** - Initial state with question
2. **Planner** - Creates plan: ["Search knowledge base", "Analyze results", "Generate answer"]
3. **Retrieve** - Searches for relevant documents
4. **Evaluate** - Checks if retrieved info answers question
5. **Generate** - Produces final answer (or loops back)

## Interpreting the Output

Notice what's returned:

- `plan`: The agent's execution strategy
- `iterations`: How many times it retried (0 = first try success)
- `final_answer`: The generated response

## Key Difference from Classic RAG

In Classic RAG:
```
Query → Retrieve → Generate → Answer
```

In Agentic RAG:
```
Query → Plan → Retrieve → Evaluate → [Retry?] → Generate → Answer
```

> The agent can recognize when it needs more information and take action!

In [7]:
result = app.invoke({
    "question": "What are the key benefits of RAG over pure LLMs?",
    "plan": [],
    "retrieved_docs": [],
    "evaluation": "",
    "final_answer": None,
    "iterations": 0
})

print("Question:", result["question"])
print("\nPlan:", result.get("plan", []))
print("\nIterations:", result.get("iterations", 0))
print("\nAnswer:", result["final_answer"])

Question: What are the key benefits of RAG over pure LLMs?

Plan: ['Search knowledge base', 'Analyze results', 'Generate answer']

Iterations: 0

Answer: content="The key benefits of RAG over pure LLMs include:\n\n1. Addressing hallucinations: RAG reduces the likelihood of hallucinations by retrieving relevant information from external knowledge bases, making the model's responses more grounded in reality.\n2. Overcoming knowledge cutoff: RAG can access a vast amount of knowledge beyond the model's training data, reducing the knowledge cutoff issue that can lead to outdated or incomplete information.\n3. Providing source attribution: RAG allows for source attribution, enabling users to understand the origin of the information provided, which is not possible with pure LLMs." additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-09T15:32:31.100405Z', 'done': True, 'done_reason': 'stop', 'total_duration': 864913417, 'load_duration': 49850500, 'prompt_eval_cou

## Visualize the Graph

In [8]:
# Display the graph structure
print(app.get_graph().draw_ascii())

           +-----------+       
           | __start__ |       
           +-----------+       
                  *            
                  *            
                  *            
            +---------+        
            | planner |        
            +---------+        
           ***        ...      
          *              .     
        **                ...  
+----------+                 . 
| retrieve |              ...  
+----------+             .     
           ***        ...      
              *      .         
               **  ..          
            +----------+       
            | evaluate |       
            +----------+       
                  .            
                  .            
                  .            
            +----------+       
            | generate |       
            +----------+       
                  *            
                  *            
                  *            
            +---------+        
        

## Try Different Queries

**What:** Testing the agent with various question types.

## Query Analysis

Notice how the agent handles different queries:

### Query 1: "How does RAG reduce hallucinations?"
- ✅ Successfully retrieved and answered
- Uses retrieved context about RAG benefits

### Query 2: "What is the difference between classic RAG and agentic RAG?"
- ⚠️ Could not answer from available context
- The sample document doesn't mention agentic RAG
- Agent honestly says it can't answer

## Why This Matters

The agent correctly identifies when it CAN'T answer:
- "the provided context does not mention agentic RAG"
- This prevents hallucinations!

## Improving the Agent

To answer the second question, you could:
1. Add more documents about agentic RAG
2. Add a web search tool
3. Enable multiple retrieval iterations
4. Add a "fallback" tool for external search

In [9]:
queries = [
    "How does RAG reduce hallucinations?",
    "What is the difference between classic RAG and agentic RAG?"
]

for q in queries:
    result = app.invoke({
        "question": q,
        "plan": [],
        "retrieved_docs": [],
        "evaluation": "",
        "final_answer": None,
        "iterations": 0
    })
    print(f"Q: {q}")
    print(f"A: {result['final_answer']}\n")

Q: How does RAG reduce hallucinations?
A: content='RAG reduces hallucinations by retrieving relevant information from external knowledge bases, which helps to ground the model\'s responses in actual facts and data, rather than relying on its own internal knowledge or generating entirely new information. This process prevents the model from "hallucinating" or making things up, as it is forced to rely on verifiable sources to inform its responses.' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-09T15:33:09.689026Z', 'done': True, 'done_reason': 'stop', 'total_duration': 593479667, 'load_duration': 51182084, 'prompt_eval_count': 169, 'prompt_eval_duration': 83012583, 'eval_count': 74, 'eval_duration': 449272794, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019cd33b-0f66-7f30-ab81-8d640018a72b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 169, 'output_tokens': 74, 'total_tokens': 243}

Q: W

## Next Steps

### Expanding the Agent

| Addition | Description |
|----------|-------------|
| **Web Search Tool** | Search the internet for current info |
| **Calculator Tool** | Perform calculations |
| **Memory** | Store conversation history |
| **Multi-Agent** | Multiple agents collaborate |
| **Error Handling** | Graceful failure recovery |

### When to Use Agentic RAG

✅ **Great for:**
- Complex questions needing multiple steps
- When retrieval might need refinement
- Multi-tool scenarios
- Questions requiring external data

❌ **Classic RAG may be better:**
- Simple, direct questions
- Latency-sensitive applications
- Lower complexity requirements

### Learning Path

1. **Notebook 1**: Classic RAG (foundations)
2. **Notebook 2**: KG-RAG (structured data)
3. **Notebook 4**: Evaluation
4. **Notebook 5**: Production deployment